In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

DATA_PROC = Path("data/processed")
DB_PATH   = Path("data/processed/munich_airbnb.db")

# Load the cleaned data
listings = pd.read_csv(DATA_PROC / "listings_clean.csv", low_memory=False)

print(f"✅ Loaded cleaned data")
print(f"Rows: {listings.shape[0]:,}")
print(f"Columns: {listings.shape[1]}")

✅ Loaded cleaned data
Rows: 5,487
Columns: 61


In [2]:
# Connect to SQLite database (creates it if it doesn't exist)
conn = sqlite3.connect(DB_PATH)

# Load listings into SQLite
listings.to_sql("listings", conn, if_exists="replace", index=False)

print(f"✅ Loaded into SQLite: {DB_PATH}")
print(f"Rows loaded: {listings.shape[0]:,}")

# Verify by reading back
test = pd.read_sql("SELECT COUNT(*) AS total_rows FROM listings", conn)
print(f"Rows in database: {test['total_rows'][0]:,}")

conn.close()
print("✅ SQLite connection closed!")

✅ Loaded into SQLite: data\processed\munich_airbnb.db
Rows loaded: 5,487
Rows in database: 5,487
✅ SQLite connection closed!


In [4]:
import mysql.connector

conn_mysql = mysql.connector.connect(
    host="localhost",
    port=3306,
    user="root",
    password="Catherine_08051999",
    database="munich_airbnb"
)

print("✅ Connected to MySQL!")

✅ Connected to MySQL!


In [5]:
import pandas as pd
from pathlib import Path

DATA_PROC = Path("data/processed")
listings = pd.read_csv(DATA_PROC / "listings_clean.csv", low_memory=False)

# Write listings to MySQL
cursor = conn_mysql.cursor()

# Drop table if exists and recreate
cursor.execute("DROP TABLE IF EXISTS listings")

# Creating the table
cursor.execute("""
    CREATE TABLE listings (
        id BIGINT,
        name TEXT,
        neighbourhood_cleansed VARCHAR(255),
        room_type VARCHAR(100),
        price FLOAT,
        minimum_nights INT,
        number_of_reviews INT,
        review_scores_rating FLOAT,
        bedrooms FLOAT,
        beds FLOAT,
        accommodates INT
    )
""")

# Insert rows
for _, row in listings[["id","name","neighbourhood_cleansed",
                          "room_type","price","minimum_nights",
                          "number_of_reviews","review_scores_rating",
                          "bedrooms","beds","accommodates"]].iterrows():
    cursor.execute("""
        INSERT INTO listings VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, tuple(row))

conn_mysql.commit()
print(f"✅ Loaded {listings.shape[0]:,} rows into MySQL!")
cursor.close()

✅ Loaded 5,487 rows into MySQL!


True